## EA Pipeline Skeleton

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import datetime
import random
import sys
import os
import glob

### PlaneModel defines a set of useful classes

In [ ]:
from  PlaneModel import job,aircraft,staff,problem

### Import Data Generator and its aircraft instaces method

In [ ]:
from generate_aircrafts_seed import generate_aircraft_instances

### Import EA Computing Functions and Depencies

In [ ]:
from EA_Model import randSol, evaluate, timeMutate, mutate, copyG, contains, xo, tour, rip

### Generate CSV file with specified input parameters

In [ ]:
generate_aircraft_instances(
    work_packages_file="work_packages.csv",  # Relative Path to WP input CSV
    output_dir="generated_aircrafts",        # Output directory name
    num_instances=10,                        # Number of CSV files to generate
    seed=20                                  # Fix the seed to 20 to ensure reproducibility
)

### Conditional Checking of Data Directory

In [ ]:
def is_directory_empty(path_directory):
    """
    Check if a directory is empty.
    """
    path = Path(path_directory)
    # Check if the path exists and is a directory
    if not path.exists() or not path.is_dir():
        raise ValueError("Invalid directory path")
    # check if the directory is empty
    return not any(path.iterdir())

In [ ]:
path_directory = "generated_aircrafts"
try:
    if is_directory_empty(path_directory):
        print("Directory is empty")
    else:
        print("sample aircrfat instances are generated")
except ValueError as e:
    print(e)
    sys.exit(1)
    

### Reading Work Packages and Technicians files to DF

In [ ]:
work_packages = pd.read_csv('work_packages.csv')
people = pd.read_csv('technicians.csv')

## Load Problem instance from the directory of database


In [ ]:
# Path to the directory containing the CSV files
db_path = './generated_aircrafts'

# Get a list of all CSV files in the directory
csv_files = glob.glob(os.path.join(db_path, '*.csv'))

In [ ]:
#create a dict to store the aircraft samples
aircraft_samples = {}

# Loop through each CSV file and read it into a DataFrame
for csv_file in csv_files:
    # Extract the filename without the directory and extension
    filename = os.path.splitext(os.path.basename(csv_file))[0]
    
    # Read the CSV file into a DataFrame
    df = pd.read_csv(csv_file)
    
    # Store the DataFrame in the dictionary with the filename as the key
    aircraft_samples[filename] = df

### Function to Calculate and Convert Time of CSV Fields

In [ ]:
def format_timedelta(td):
    total_seconds = td.total_seconds()
    total_minutes = round(total_seconds / 60)
    hours, minutes = divmod(total_minutes, 60)
    return f"{int(hours):02d}:{int(minutes):02d}"

In [ ]:
# Combine the aircraft samples into a single DataFrame
combined_df = pd.concat(aircraft_samples.values(), ignore_index=True)

In [ ]:
# Convert Turn Around Time to timedelta
combined_df['Turn Around Time'] = pd.to_timedelta(
    combined_df['Turn Around Time'].str.replace(':', 'h ') + 'm'
)

# Calculate the average and total Turn Around Time for each aircraft
total_tat = combined_df.groupby('Aicraft (A/C) Serial Number')['Turn Around Time'].sum()
average_tat = combined_df.groupby('Aicraft (A/C) Serial Number')['Turn Around Time'].mean()

In [ ]:
# Count work items and calculate average
combined_df['Work Count'] = combined_df['Work that needs to be carried out'].str.split(', ').apply(len)
average_work_count = combined_df.groupby('Aicraft (A/C) Serial Number')['Work Count'].mean()
average_work_count = average_work_count.astype(int)

In [ ]:
# Format results into a DataFrame
results_df = pd.DataFrame({
    'Total Turn Around Time': total_tat,
    'Average Turn Around Time': average_tat,
    'Average Work Count': average_work_count
}).reset_index()

results_df['Total Turn Around Time'] = results_df['Total Turn Around Time'].apply(format_timedelta)
results_df['Average Turn Around Time'] = results_df['Average Turn Around Time'].apply(format_timedelta)
results_df['Average Work Count'] = results_df['Average Work Count'].astype(int)

In [ ]:
# mapping work pakcage numbers to the minitues and personnel and create a dict to store the WP numbers with Minutes
wp_to_min = work_packages.set_index('WP number')['Minutes'].to_dict()

wp_to_personnel = dict(zip(work_packages['WP number'], work_packages['Number of Personnel']))

In [ ]:
# Split the 'Work that needs to be carried out' into a list of WPs and clean whitespace
combined_df['Work that needs to be carried out'] = combined_df['Work that needs to be carried out'].apply(lambda x: [wp.strip() for wp in x.split(', ')])

# Calculate total minutes and personnel for each aircraft
combined_df['Total Minutes'] = combined_df['Work that needs to be carried out'].apply(lambda wps: sum(wp_to_min.get(wp, 0) for wp in wps))

combined_df['Total Personnel'] = combined_df['Work that needs to be carried out'].apply(lambda wps: sum(wp_to_personnel.get(wp, 0) for wp in wps))

In [ ]:
# Extract relevant columns
output_df = combined_df[['Aicraft (A/C) Serial Number', 'Total Minutes', 'Total Personnel']]

In [ ]:
merged_df = pd.merge(results_df, output_df,
    left_on='Aicraft (A/C) Serial Number',  # column name in results_df
    right_on='Aicraft (A/C) Serial Number',  # column name in output_df
    how='outer'  # use outer join to keep all rows from both DataFrames
)

In [ ]:
# Create logs directory if it doesn't exist
os.makedirs('logs', exist_ok=True)

output_path = os.path.join('logs', 'aircraft_work_summary.csv')
merged_df.to_csv(output_path, index=False)
print(f"Aircraft work summary saved to {output_path}")

### Call the EA Model and Populate it with DataFrames of Aircraft Samples

#### Fine-tuning the EA outputs

In [ ]:
for k , planes in aircraft_samples.items():
    instance = problem(people,planes,work_packages)
    
    print(f"Running for {k} instance")

    fitness_All = []
    for run in range(10):
        pop_size = 1500 
        budget = 100000

        best = None

        pop = []
        for c in range(pop_size):
            
            i = randSol(instance)
            f= evaluate(i, instance)[0]
            p = (f, i)
            pop.append(p)
            if len(pop) == 1:
                best = p
            if f < best[0]:
                best = p

        print(f"Run {run + 1} - Init")
        print(f"Initial best: {best[0]}")

        evals = 0
        
        while evals < budget:
            evals += 1
            if random.choice([True, False]):
                parent = tour(pop)
                ng = copyG(parent[1])
            else:
                ng = xo(tour(pop)[1], tour(pop)[1])
            
            mutate(ng, instance)
            nf = evaluate(ng, instance)[0]
            child = (nf, ng)

            toGo = rip(pop)
            if toGo[0] > child[0]:
                pop.remove(toGo)
                pop.append(child)
                if child[0] < best[0]:
                    best = (child[0], copyG(child[1]))
                    instance.reset()
                    r = evaluate(best[1], instance)
                    print(f"Evals: {evals}")
                    print(f"Improved ({r[0]}) Missing staff = {r[1]} Late = {r[2]} Late mins = {r[3]}")

        fitness_All.append(best[0])
        print(f"Run {run + 1} - Done: {best[0]}\n")
        
    # compute the average of fitness values in the population for 10 iterations
    np_avg_fitness = np.mean(fitness_All)
    avg_fitness = sum(fitness_All) / len(fitness_All)
    print(f"Average fitness over 10 runs: Math Method {avg_fitness} and Numpy Method {np_avg_fitness}")

    # compute standard deviation of fitness values in the population for 10 iterations
    std_dev = np.std(fitness_All)
    print(f"Standard Deviation: {std_dev}")